# Single Event Multi-Parameter Loss Scan
## Analyzing combined_product_loss sensitivity to all parameters for one event

In [ ]:
import sys
sys.path.append('..')

from lucid.geometry import generate_detector
from lucid.utils import load_single_event, save_single_event, generate_random_params, print_particle_params
from lucid.simulation import setup_event_simulator
from lucid.generate import read_photon_data_from_photonsim
from lucid.utils import spherical_to_cartesian
from lucid.detector_params import ParticleParams, load_detector_params

import jax
import jax.numpy as jnp
import time

from jax import jit
from pathlib import Path

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

import numpy as np
from functools import partial
import pickle
from tqdm import tqdm
from jax import grad, jit, vmap, value_and_grad
import uproot
from scipy.interpolate import interp1d

## Setup Detector and Constants

In [ ]:
# Configuration
default_json_filename = '../config/SK_geom_config.json'
PHYSICS_CONFIG = '../config/SK_physics_config.json'
data_file =  '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'
TEMPERATURE = 0.10  # Fixed temperature
N_SCAN_POINTS = 21  # Number of scan points per parameter
K = 7
Nphot = 150_000

# Speed of light in medium
C_MEDIUM = 0.299792/1.33  # speed of light in medium

# Setup detector
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)


# Setup data simulator for generating target events (is_data=True, temperature=0.0)
data_simulator = setup_event_simulator(default_json_filename, Nphot, temperature=0.0, K=20,
                                      is_data=True, is_calibration=False,
                                      physics_config=PHYSICS_CONFIG, default_detector_params=True)

# Setup prediction simulator with fixed temperature (is_data=False)
prediction_simulator = setup_event_simulator(default_json_filename, Nphot, TEMPERATURE, max_sensors_per_cell=4, K=K, is_data=False,
                                            physics_config=PHYSICS_CONFIG, default_detector_params=True, hit_mode='aggregated')

print(f"Number of detectors: {NUM_DETECTORS}")
print(f"Speed of light in medium: {C_MEDIUM}")
print(f"Number of scan points per parameter: {N_SCAN_POINTS}")

# Load ROOT file information
with uproot.open(data_file) as file:
    tree = file['OpticalPhotons']
    n_entries = tree.num_entries
print(f"ROOT file has {n_entries} entries")

## Detector Parameters

## Combined Product Loss Function

In [ ]:
from lucid.losses import counts_loss, origin_time_loss, cone_time_loss

import jax
import jax.numpy as jnp
from jax import jit

@jit
def combined_product_loss(params, hit_detector_positions, observed_times, observed_counts, 
                                    true_data, key):
    """
    Combined loss function: product of vertex loss, counts loss, and energy loss
    
    Args:
        params: [x, y, z, t0, theta, phi, energy] where theta and phi are spherical direction angles
        vertex_weight: scaling factor for vertex loss contribution
        counts_weight: scaling factor for counts loss contribution  
        energy_weight: scaling factor for energy loss contribution
    """
    position = params[:3]
    t0 = params[3]
    theta = params[4]
    phi = params[5]
    energy = params[6]

    track = ParticleParams(energy=energy, position=position, theta=theta, phi=phi, t0=jnp.array(0.0))
    simulated_data = prediction_simulator(track, key)
    simulated_counts = simulated_data[0]
    simulated_time = simulated_data[1]

    # Calculate individual loss components
    vertex_loss_val = origin_time_loss(position, hit_detector_positions, observed_times,
                                      observed_counts, t0)
    counts_loss_val = counts_loss(observed_counts, simulated_counts)
    time_loss_val = cone_time_loss(observed_counts, simulated_time, observed_times, t0, tau=0.23)

    # Convert spherical to cartesian for direction loss
    direction = spherical_to_cartesian(theta, phi)

    combined_loss = jnp.sqrt((vertex_loss_val + 1e-6) * (counts_loss_val + 1e-6) * (time_loss_val + 1e-6) )
    return combined_loss
    
print("Combined product loss function defined")

## Generic Parameter Scan Function

In [ ]:
def perform_parameter_scan(true_params, true_data, key, param_name, param_idx, scan_range, 
                          true_param_values, use_relative=True, verbose=False):
    """
    Perform 1D scan over a single parameter using combined product loss function

    Args:
        true_params: ParticleParams object
        true_data: target data-like event (hit_counts, hit_times)
        key: JAX random key
        param_name: name of parameter being scanned (for display)
        param_idx: index in params array [x, y, z, t0, theta, phi, energy]
        scan_range: range to scan (relative ± or absolute range)
        true_param_values: [x, y, z, t0, theta, phi, energy] array of true values
        use_relative: if True, scan_range is relative to true value; if False, absolute
        verbose: if True, print detailed timing information

    Returns:
        dict with param_values, losses, gradients, and timing info
    """
    t_start_total = time.time()

    # Setup phase
    t_start_setup = time.time()

    # Extract hit times from true_data
    hit_counts, hit_times = true_data

    # Filter for detectors that were hit
    hit_mask = hit_counts > -1
    hit_detector_positions = detector_points[hit_mask]
    observed_times = hit_times[hit_mask]
    observed_charge = hit_counts[hit_mask]

    # Generate scan points
    true_value = true_param_values[param_idx]
    if use_relative:
        param_values = jnp.linspace(true_value - scan_range, true_value + scan_range, N_SCAN_POINTS)
    else:
        param_values = jnp.linspace(scan_range[0], scan_range[1], N_SCAN_POINTS)

    t_setup = time.time() - t_start_setup

    # Define loss and gradient function
    def loss_and_grad_fn(params):
        def loss_fn(p):
            # Need a random key for simulator in combined_product_loss
            loss_key = jax.random.PRNGKey(42)  # Fixed key for consistency
            return combined_product_loss(p, hit_detector_positions, observed_times, observed_charge,
                                        true_data, loss_key)
        return value_and_grad(loss_fn)(params)

    # Warmup (JIT compilation) - run once before timing
    t_start_warmup = time.time()
    warmup_params = jnp.array(true_param_values)
    _ = loss_and_grad_fn(warmup_params)
    jax.block_until_ready(_)  # Wait for computation to complete
    t_warmup = time.time() - t_start_warmup

    # Scan loop
    losses = []
    gradients = []
    per_point_times = []

    t_start_scan = time.time()
    for i, param_val in enumerate(param_values):
        t_point_start = time.time()

        # Create parameter vector with modified parameter
        params = jnp.array(true_param_values)
        params = params.at[param_idx].set(param_val)

        # Calculate loss and gradient
        t_loss_grad_start = time.time()
        loss, grad_val = loss_and_grad_fn(params)
        jax.block_until_ready((loss, grad_val))  # Ensure computation completes
        t_loss_grad = time.time() - t_loss_grad_start

        # Extract gradient for this parameter
        param_gradient = grad_val[param_idx]

        losses.append(loss)
        gradients.append(param_gradient)

        t_point = time.time() - t_point_start
        per_point_times.append(t_point)

        if verbose and i == 0:
            print(f"    First point timing: {t_point:.4f}s (loss+grad: {t_loss_grad:.4f}s)")

    t_scan = time.time() - t_start_scan
    t_total = time.time() - t_start_total

    # Timing statistics
    per_point_times = np.array(per_point_times)
    timing_info = {
        'total_time': t_total,
        'setup_time': t_setup,
        'warmup_time': t_warmup,
        'scan_time': t_scan,
        'avg_per_point': np.mean(per_point_times),
    }

    if verbose:
        print(f"\n  {param_name} scan timing:")
        print(f"    Total: {timing_info['total_time']:.4f}s")
        print(f"    Per point (avg): {timing_info['avg_per_point']:.4f}s")

    return {
        'param_name': param_name,
        'param_values': jnp.array(param_values),
        'losses': jnp.array(losses),
        'gradients': jnp.array(gradients),
        'true_value': true_value,
        'n_hit_detectors': jnp.sum(hit_mask),
        'timing': timing_info
    }

print("Generic parameter scan function defined")

## Generate Single Data-Like Event

In [ ]:
entry_idx = 2

# Load photon data from ROOT file
photon_data = read_photon_data_from_photonsim(data_file, entry_idx)

# Process photon data
photon_origins = photon_data['photon_origins']
photon_directions = photon_data['photon_directions']
photon_times = photon_data['photon_times']
N = len(photon_origins)
# photon_data['rotation_axis'] = jnp.array([0.0, 0.0, 1.0])
# photon_data['rotation_angle'] = jnp.array(0.0)
# photon_data['apply_rotation'] = jnp.array(False)

# the number 1_000_000 is hard coded also in _simulation_core
padding_size = max(0, 1_000_000-N)

# Pad the origins array (2D array with shape [N,3])
photon_data['photon_origins'] = jnp.pad(photon_origins, ((0, padding_size), (0, 0)), 
                                    mode='constant', constant_values=0)

# Pad the directions array with a default unit vector [0,0,1]
default_direction = jnp.array([0.0, 0.0, 1.0])
padding_directions = jnp.tile(default_direction, (padding_size, 1))
if padding_size > 0:
    photon_data['photon_directions'] = jnp.concatenate([photon_directions, padding_directions], axis=0)
else:
    photon_data['photon_directions'] = photon_directions

# Pad the times array (1D array with shape [N])
photon_data['photon_times'] = jnp.pad(photon_times, (0, padding_size),
                                      mode='constant', constant_values=0)

photon_data['N'] = N

# Generate random track parameters
key = jax.random.PRNGKey(44)

# Random position within detector bounds (60% of full volume)
fraction = 0.6
r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=detector.r * fraction)
key, _ = jax.random.split(key)
theta_pos = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
key, _ = jax.random.split(key)
z_vert = jax.random.uniform(key, shape=(), minval=-detector.H/2 * fraction, 
                           maxval=detector.H/2 * fraction)
true_position = jnp.array([r_vert * jnp.cos(theta_pos), r_vert * jnp.sin(theta_pos), z_vert])

# Random direction
key, _ = jax.random.split(key)
phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
key, _ = jax.random.split(key)
cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
sin_theta = jnp.sqrt(1 - cos_theta**2)
true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

original_direction = jnp.array([0.0, 0.0, 1.0])
true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)

# Rotation axis = cross product of original and target directions
rotation_axis = jnp.cross(original_direction, true_direction_norm)
axis_norm = jnp.linalg.norm(rotation_axis)

# Handle case where directions are parallel (axis_norm ~ 0)
rotation_axis = jnp.where(
    axis_norm < 1e-6,
    jnp.array([1.0, 0.0, 0.0]),  # Arbitrary axis when parallel
    rotation_axis / (axis_norm + 1e-8)
)

# Rotation angle = arccos of dot product
rotation_angle = jnp.arccos(jnp.clip(
    jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
))

# Set rotation parameters
photon_data['rotation_axis'] = rotation_axis
photon_data['rotation_angle'] = rotation_angle
photon_data['apply_rotation'] = jnp.array(True)

# Set translation parameters to move from origin to true_position
photon_data['apply_translation'] = jnp.array(True)
photon_data['translation_vector'] = true_position


# Use energy from ROOT file
true_energy = photon_data['energy']  # Fixed energy

# Create particle parameters
true_track = ParticleParams.from_cartesian(energy=true_energy, position=true_position, direction=true_direction, t0=0.0)

# Generate data-like event
key, _ = jax.random.split(key)
true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))

# Convert direction to spherical coordinates
from lucid.optimization.utils.functions import cartesian_to_spherical
true_theta, true_phi = cartesian_to_spherical(true_direction)

# Use t0=0 as default
true_t0 = 0.0

print(f"\nGenerated single event:")
print(f"  Energy: {true_energy:.2f} MeV")
print(f"  Position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}] m")
print(f"  Direction (Cartesian): [{true_direction[0]:.3f}, {true_direction[1]:.3f}, {true_direction[2]:.3f}]")
print(f"  Direction (Spherical): theta={true_theta:.3f} rad, phi={true_phi:.3f} rad")
print(f"  t0: {true_t0:.2f} ns")

# Count hit detectors
hit_counts, hit_times = true_data
n_hit = jnp.sum(hit_counts > -1)
print(f"  Hit detectors: {n_hit}")

## Perform Parameter Scans

In [ ]:
true_param_values = [
    float(true_position[0]),  # x
    float(true_position[1]),  # y
    float(true_position[2]),  # z
    float(true_t0),           # t0
    float(true_theta),        # theta
    float(true_phi),          # phi
    float(true_energy)        # energy
]

# Define scan configurations
# Format: (param_name, param_idx, scan_range, use_relative)
scan_configs = [
    ('X', 0, 0.5, True),           # ±0.5 m around true X
    ('Y', 1, 0.5, True),           # ±0.5 m around true Y
    ('Z', 2, 0.5, True),           # ±0.5 m around true Z
    ('t0', 3, 5.0, True),         # ±10 ns around true t0
    ('theta', 4, 0.3, True),       # ±0.3 rad around true theta
    ('phi', 5, 0.3, True),         # ±0.3 rad around true phi
    ('E', 6, 100.0, True),         # ±200 MeV around true energy
]

# Perform all scans
scan_results = []

print(f"\nPerforming {len(scan_configs)} parameter scans...")
print("=" * 60)

for i, (param_name, param_idx, scan_range, use_relative) in enumerate(scan_configs):
    print(f"\nScanning parameter: {param_name} (index {param_idx})")
    
    key, _ = jax.random.split(key)
    result = perform_parameter_scan(
        true_track, 
        true_data, 
        key, 
        param_name, 
        param_idx, 
        scan_range,
        true_param_values,
        use_relative=use_relative,
        verbose=True
    )
    
    scan_results.append(result)

print("\n" + "=" * 60)
print("All parameter scans completed!")

## Visualize All Parameter Scans

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(4, 4, figsize=(10, 8))
axes = axes.flatten()

all_handles = []
all_labels = []

# Plot loss and gradient for each parameter
for i, result in enumerate(scan_results):
    param_name = result['param_name']
    param_values = np.array(result['param_values'])
    losses = np.array(result['losses'])
    gradients = np.array(result['gradients'])
    true_value = result['true_value']
    
    # Plot loss
    ax_loss = axes[2*i]
    line_loss, = ax_loss.plot(param_values, losses, 'b-', linewidth=2, label='Loss')
    true_line_loss = ax_loss.axvline(true_value, color='red', linestyle='--', alpha=0.7, label='True Value')
    min_loss_idx = np.argmin(losses)
    min_loss_val = param_values[min_loss_idx]
    min_loss_line = ax_loss.axvline(min_loss_val, color='orange', linestyle=':', alpha=0.7, label='Min Loss')
    ax_loss.plot(min_loss_val, losses[min_loss_idx], 'o', color='orange', markersize=8)
    
    ax_loss.set_title(f'{param_name} - Loss', fontsize=12, fontweight='bold')
    ax_loss.set_xlabel(param_name)
    ax_loss.set_ylabel('Loss')
    ax_loss.grid(True, alpha=0.3)
    
    # Plot gradient
    ax_grad = axes[2*i + 1]
    line_grad, = ax_grad.plot(param_values, gradients, 'g-', linewidth=2, label='Gradient')
    true_line_grad = ax_grad.axvline(true_value, color='red', linestyle='--', alpha=0.7, label='True Value')
    zero_grad_line = None
    ax_grad.axhline(0, color='gray', linestyle=':', alpha=0.5)

    if np.min(gradients) <= 0 <= np.max(gradients):
        for j in range(len(gradients) - 1):
            g1, g2 = gradients[j], gradients[j + 1]
            if g1 == 0:
                zero_grad_val = param_values[j]
                break
            elif g1 * g2 < 0:
                x1, x2 = param_values[j], param_values[j + 1]
                zero_grad_val = x1 - g1 * (x2 - x1) / (g2 - g1)
                break
        zero_grad_line = ax_grad.axvline(zero_grad_val, color='purple', linestyle='-.', alpha=0.7, label='Zero Grad')
    
    ax_grad.set_title(f'{param_name} - Gradient', fontsize=12, fontweight='bold')
    ax_grad.set_xlabel(param_name)
    ax_grad.set_ylabel(f'∂Loss/∂{param_name}')
    ax_grad.grid(True, alpha=0.3)

    # Collect handles/labels only once
    if i == 0:
        # Collect all relevant handles and labels
        for item in [line_loss, true_line_loss, min_loss_line, line_grad, true_line_grad]:
            all_handles.append(item)
            all_labels.append(item.get_label())
        if zero_grad_line:
            all_handles.append(zero_grad_line)
            all_labels.append(zero_grad_line.get_label())

# Hide unused subplot
axes[-2].axis('off')
axes[-1].axis('off')

# Create a single, figure-level legend
fig.legend(
    all_handles, all_labels,
    loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.75, 0.2)
)

plt.tight_layout(rect=[0, 0.0, 1, 1])  # Leave space at bottom for legend
plt.show()


## Summary Statistics

In [ ]:
print("=" * 80)
print("SINGLE EVENT MULTI-PARAMETER SCAN SUMMARY")
print("=" * 80)

print(f"\nEvent Details:")
print(f"  Energy: {true_energy:.2f} MeV")
print(f"  Position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}] m")
print(f"  Direction: theta={true_theta:.3f} rad, phi={true_phi:.3f} rad")
print(f"  Hit detectors: {n_hit}")

print(f"\nParameter Scan Results:")
print("-" * 80)
print(f"{'Parameter':<10} {'True Value':<15} {'Min Loss Value':<15} {'Delta':<12} {'Zero Grad?':<12}")
print("-" * 80)

for result in scan_results:
    param_name = result['param_name']
    param_values = np.array(result['param_values'])
    losses = np.array(result['losses'])
    gradients = np.array(result['gradients'])
    true_value = result['true_value']
    
    # Find minimum loss position
    min_loss_idx = np.argmin(losses)
    min_loss_val = param_values[min_loss_idx]
    delta = min_loss_val - true_value
    
    # Check for zero crossing
    has_zero_crossing = np.min(gradients) <= 0 <= np.max(gradients)
    
    print(f"{param_name:<10} {true_value:<15.4f} {min_loss_val:<15.4f} {delta:<12.4f} {'Yes' if has_zero_crossing else 'No':<12}")

print("-" * 80)

# Timing summary
print(f"\nTiming Summary:")
total_time = sum([result['timing']['total_time'] for result in scan_results])
print(f"  Total scan time: {total_time:.2f}s ({total_time/60:.2f} min)")
for result in scan_results:
    param_name = result['param_name']
    scan_time = result['timing']['total_time']
    avg_per_point = result['timing']['avg_per_point']
    print(f"  {param_name}: {scan_time:.2f}s (avg per point: {avg_per_point:.4f}s)")

print("\n" + "=" * 80)

# 1D Gradient analysis for the X coordinate for Multiple Data-like Events

In [ ]:
N_EVENTS = 25  # Number of data-like events to analyze

In [ ]:
def perform_x_position_scan_combined_loss(true_track, true_data, key, x_scan_range=0.5, verbose=False):
    """
    Perform 1D scan over X-position using combined product loss function

    Args:
        true_track: ParticleParams object
        true_data: target data-like event (hit_counts, hit_times)
        key: JAX random key
        x_scan_range: range to scan around true X position (±range)
        verbose: if True, print detailed timing information

    Returns:
        dict with x_positions, losses, gradients, and timing info
    """
    t_start_total = time.time()

    # Setup phase
    t_start_setup = time.time()
    true_energy, true_position, true_direction = true_track.energy, true_track.position, true_track.direction
    true_x = true_position[0]

    # Extract hit times from true_data
    hit_counts, hit_times = true_data

    # Filter for detectors that were hit
    hit_mask = hit_counts > -1
    hit_detector_positions = detector_points[hit_mask]
    observed_times = hit_times[hit_mask]
    observed_charge = hit_counts[hit_mask]

    # Generate X-position scan points
    x_positions = jnp.linspace(true_x - x_scan_range, true_x + x_scan_range, N_SCAN_POINTS)

    # Convert true direction to spherical coordinates
    from lucid.optimization.utils.functions import cartesian_to_spherical
    true_theta, true_phi = cartesian_to_spherical(true_direction)

    # Use t0=0 as default (can be adjusted)
    t0 = 0.0
    t_setup = time.time() - t_start_setup

    # Define loss and gradient function
    def loss_and_grad_fn(params):
        def loss_fn(p):
            # Need a random key for simulator in combined_product_loss
            loss_key = jax.random.PRNGKey(42)  # Fixed key for consistency
            return combined_product_loss(p, hit_detector_positions, observed_times, observed_charge,
                                        true_data, loss_key)
        return value_and_grad(loss_fn)(params)

    # Warmup (JIT compilation) - run once before timing
    t_start_warmup = time.time()
    warmup_params = jnp.array([true_x, true_position[1], true_position[2], t0, true_theta, true_phi, true_energy])
    _ = loss_and_grad_fn(warmup_params)
    jax.block_until_ready(_)  # Wait for computation to complete
    t_warmup = time.time() - t_start_warmup

    # Scan loop
    losses = []
    gradients = []
    per_point_times = []

    t_start_scan = time.time()
    for i, x_pos in enumerate(x_positions):
        t_point_start = time.time()

        # Create parameter vector with modified X coordinate
        # params: [x, y, z, t0, theta, phi, energy]
        params = jnp.array([
            x_pos,  # modified x
            true_position[1],  # y
            true_position[2],  # z
            t0,  # t0
            true_theta,  # theta
            true_phi,  # phi
            true_energy  # energy
        ])

        # Calculate loss and gradient
        t_loss_grad_start = time.time()
        loss, grad_val = loss_and_grad_fn(params)
        jax.block_until_ready((loss, grad_val))  # Ensure computation completes
        t_loss_grad = time.time() - t_loss_grad_start

        # Extract X-position gradient
        x_gradient = grad_val[0]  # gradient w.r.t. params[0] (x position)

        losses.append(loss)
        gradients.append(x_gradient)

        t_point = time.time() - t_point_start
        per_point_times.append(t_point)

        if verbose and i == 0:
            print(f"    First point timing: {t_point:.4f}s (loss+grad: {t_loss_grad:.4f}s)")

    t_scan = time.time() - t_start_scan
    t_total = time.time() - t_start_total

    # Timing statistics
    per_point_times = np.array(per_point_times)
    timing_info = {
        'total_time': t_total,
        'setup_time': t_setup,
        'warmup_time': t_warmup,
        'scan_time': t_scan,
        'avg_per_point': np.mean(per_point_times),
        'std_per_point': np.std(per_point_times),
        'min_per_point': np.min(per_point_times),
        'max_per_point': np.max(per_point_times),
        'first_point_time': per_point_times[0],
        'subsequent_avg': np.mean(per_point_times[1:]) if len(per_point_times) > 1 else 0,
    }

    if verbose:
        print(f"\n  Timing breakdown:")
        print(f"    Setup:              {timing_info['setup_time']:.4f}s")
        print(f"    Warmup (JIT):       {timing_info['warmup_time']:.4f}s")
        print(f"    Scan loop:          {timing_info['scan_time']:.4f}s")
        print(f"    Total:              {timing_info['total_time']:.4f}s")
        print(f"    Per point (avg):    {timing_info['avg_per_point']:.4f}s \u00b1 {timing_info['std_per_point']:.4f}s")
        print(f"    First point:        {timing_info['first_point_time']:.4f}s")
        print(f"    Subsequent (avg):   {timing_info['subsequent_avg']:.4f}s")

    return {
        'x_positions': jnp.array(x_positions),
        'losses': jnp.array(losses),
        'gradients': jnp.array(gradients),
        'n_hit_detectors': jnp.sum(hit_mask),
        'timing': timing_info
    }

print("X-position scan function defined (with timing)")

In [ ]:
all_results = {
    'c_medium': C_MEDIUM,
    'temperature': TEMPERATURE,
    'n_events': N_EVENTS,
    'n_scan_points': N_SCAN_POINTS,
    'events': [],
    'timing_stats': []  # Store timing for each event
}

print(f"Analyzing {N_EVENTS} data-like events with combined product loss X-position scans...")

# Generate random keys for each event
main_key = jax.random.PRNGKey(42)
event_keys = jax.random.split(main_key, N_EVENTS)

for event_idx in tqdm(range(N_EVENTS), desc="Processing events"):
    # Select entry from ROOT file (cycling through available entries)
    entry_idx = event_idx % n_entries
    
    # Load photon data from ROOT file
    photon_data = read_photon_data_from_photonsim(data_file, entry_idx)
    #photon_data['N'] = len(photon_data['photon_origins'])

    # Process photon data
    photon_origins = photon_data['photon_origins']
    photon_directions = photon_data['photon_directions']
    photon_times = photon_data['photon_times']
    N = len(photon_origins)

    # the number 1_000_000 is hard coded also in _simulation_core
    padding_size = max(0, 1_000_000-N)

    # Pad the origins array (2D array with shape [N,3])
    photon_data['photon_origins'] = jnp.pad(photon_origins, ((0, padding_size), (0, 0)), 
                                        mode='constant', constant_values=0)

    # Pad the directions array with a default unit vector [0,0,1]
    default_direction = jnp.array([0.0, 0.0, 1.0])
    padding_directions = jnp.tile(default_direction, (padding_size, 1))
    if padding_size > 0:
        photon_data['photon_directions'] = jnp.concatenate([photon_directions, padding_directions], axis=0)
    else:
        photon_data['photon_directions'] = photon_directions

    # Pad the times array (1D array with shape [N])
    photon_data['photon_times'] = jnp.pad(photon_times, (0, padding_size),
                                          mode='constant', constant_values=0)

    photon_data['N'] = N
    
    # Generate random track parameters
    key = event_keys[event_idx]
    
    # Random position within detector bounds (60% of full volume)
    fraction = 0.6
    r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=detector.r * fraction)
    key, _ = jax.random.split(key)
    theta = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    z_vert = jax.random.uniform(key, shape=(), minval=-detector.H/2 * fraction, 
                               maxval=detector.H/2 * fraction)
    true_position = jnp.array([r_vert * jnp.cos(theta), r_vert * jnp.sin(theta), z_vert])
    
    # Random direction
    key, _ = jax.random.split(key)
    phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
    sin_theta = jnp.sqrt(1 - cos_theta**2)
    true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

    original_direction = jnp.array([0.0, 0.0, 1.0])
    true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)
    
    # Rotation axis = cross product of original and target directions
    rotation_axis = jnp.cross(original_direction, true_direction_norm)
    axis_norm = jnp.linalg.norm(rotation_axis)
    
    # Handle case where directions are parallel (axis_norm ~ 0)
    rotation_axis = jnp.where(
        axis_norm < 1e-6,
        jnp.array([1.0, 0.0, 0.0]),  # Arbitrary axis when parallel
        rotation_axis / (axis_norm + 1e-8)
    )
    
    # Rotation angle = arccos of dot product
    rotation_angle = jnp.arccos(jnp.clip(
        jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
    ))
    
    # Set rotation parameters
    photon_data['rotation_axis'] = rotation_axis
    photon_data['rotation_angle'] = rotation_angle
    photon_data['apply_rotation'] = jnp.array(True)
    
    # Set translation parameters to move from origin to true_position
    photon_data['apply_translation'] = jnp.array(True)
    photon_data['translation_vector'] = true_position
    
    # Use energy from ROOT file
    true_energy = photon_data['energy']  # Fixed energy
    
    # Create particle parameters
    true_track = ParticleParams.from_cartesian(energy=true_energy, position=true_position, direction=true_direction, t0=0.0)
    
    # Generate data-like event
    key, _ = jax.random.split(key)
    before_data_calc = time.time()
    true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))
    #print('data generation time: ', time.time()-before_data_calc)
    
    # Perform X-position scan with combined product loss
    # Enable verbose for first event only
    key, _ = jax.random.split(key)
    scan_results = perform_x_position_scan_combined_loss(true_track, true_data, key, verbose=(event_idx < 3))
    
    # Store results for this event
    event_result = {
        'event_idx': event_idx,
        'entry_idx': entry_idx,
        'true_energy': float(true_energy),
        'true_position': [float(x) for x in true_position],
        'true_direction': [float(x) for x in true_direction],
        'x_positions': [float(x) for x in scan_results['x_positions']],
        'losses': [float(x) for x in scan_results['losses']],
        'gradients': [float(x) for x in scan_results['gradients']],
        'n_hit_detectors': int(scan_results['n_hit_detectors'])
    }
    
    all_results['events'].append(event_result)
    all_results['timing_stats'].append(scan_results['timing'])
    
    if event_idx == 0:
        print(f"\nEvent {event_idx} (entry {entry_idx}):")
        print(f"  Energy: {true_energy:.2f} MeV")
        print(f"  Position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}] m")
        print(f"  Direction: [{true_direction[0]:.3f}, {true_direction[1]:.3f}, {true_direction[2]:.3f}]")
        print(f"  X scan range: [{scan_results['x_positions'][0]:.2f}, {scan_results['x_positions'][-1]:.2f}] m")
        print(f"  Hit detectors: {scan_results['n_hit_detectors']}")

print(f"\nCompleted analysis of {N_EVENTS} events with {N_SCAN_POINTS} X-position points each.")

In [ ]:
delta_x_values = []

for event in all_results['events']:
    x_positions = np.array(event['x_positions'])
    gradients = np.array(event['gradients'])
    true_x = event['true_position'][0]

    zero_cross_found = False

    for i in range(len(gradients) - 1):
        g1, g2 = gradients[i], gradients[i + 1]

        # Check for a sign change (crossing zero)
        if g1 == 0:
            zero_grad_x = x_positions[i]
            zero_cross_found = True
            break
        elif g1 * g2 < 0:  # gradient crosses zero between these points
            x1, x2 = x_positions[i], x_positions[i + 1]

            # Linear interpolation: x_zero = x1 - g1 * (x2 - x1) / (g2 - g1)
            zero_grad_x = x1 - g1 * (x2 - x1) / (g2 - g1)
            zero_cross_found = True
            break

    if zero_cross_found:
        delta_x = zero_grad_x - true_x
        delta_x_values.append(delta_x)
    else:
        delta_x_values.append(np.nan)

# Calculate statistics
delta_x_array = np.array(delta_x_values)
valid_delta_x = delta_x_array[~np.isnan(delta_x_array)]
mean_delta_x = np.mean(valid_delta_x)
sigma_delta_x = np.std(valid_delta_x)

print(f"Number of events with zero-crossing gradients: {len(valid_delta_x)}")
print(f"Number of events without zero-crossing gradients: {len(delta_x_array) - len(valid_delta_x)}")
print(f"Mean delta_X: {mean_delta_x:.4f} m")
print(f"Sigma delta_X: {sigma_delta_x:.4f} m")


In [ ]:
# Plot results for first few events
n_plot_events = min(10, N_EVENTS)

fig, axes = plt.subplots(2, n_plot_events, figsize=(4*n_plot_events, 8))
if n_plot_events == 1:
    axes = axes.reshape(2, 1)

for i in range(n_plot_events):
    event = all_results['events'][i]
    x_positions = np.array(event['x_positions'])
    losses = np.array(event['losses'])
    gradients = np.array(event['gradients'])
    true_x = event['true_position'][0]
    n_hit = event['n_hit_detectors']
    
    # Plot loss
    axes[0, i].plot(x_positions, losses, 'b-', linewidth=2)
    axes[0, i].axvline(true_x, color='red', linestyle='--', alpha=0.7, label='True X')
    
    # Find and mark minimum loss position
    min_loss_idx = np.argmin(losses)
    min_loss_x = x_positions[min_loss_idx]
    axes[0, i].axvline(min_loss_x, color='orange', linestyle=':', alpha=0.7, label='Min Loss')
    
    axes[0, i].set_title(f'Event {i} - Combined Product Loss\n({n_hit} hit detectors)')
    axes[0, i].set_xlabel('X Position (m)')
    axes[0, i].set_ylabel('Loss')
    axes[0, i].grid(True, alpha=0.3)
    axes[0, i].legend()
    
    # Plot gradient
    axes[1, i].plot(x_positions, gradients, 'g-', linewidth=2)
    axes[1, i].axvline(true_x, color='red', linestyle='--', alpha=0.7, label='True X')
    axes[1, i].axhline(0, color='gray', linestyle=':', alpha=0.5)
    
    # Mark zero gradient crossing if it exists
    if not np.isnan(delta_x_values[i]):
        zero_grad_x = true_x + delta_x_values[i]
        axes[1, i].axvline(zero_grad_x, color='purple', linestyle='-.', alpha=0.7, label='Zero Grad')
    
    axes[1, i].set_title(f'Event {i} - Gradient')
    axes[1, i].set_xlabel('X Position (m)')
    axes[1, i].set_ylabel('∂Loss/∂X')
    axes[1, i].grid(True, alpha=0.3)
    axes[1, i].legend()

plt.tight_layout()
plt.show()

print(f"Plotted results for first {n_plot_events} events.")

In [ ]:
# Plot histogram of delta_X values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Histogram of delta_X
ax1.hist(valid_delta_x, bins=25, alpha=0.7, edgecolor='black')
ax1.axvline(mean_delta_x, color='red', linestyle='--', label=f'Mean: {mean_delta_x:.4f} m')
ax1.axvline(mean_delta_x + sigma_delta_x, color='orange', linestyle=':', label=f'+σ: {sigma_delta_x:.4f} m')
ax1.axvline(mean_delta_x - sigma_delta_x, color='orange', linestyle=':', label=f'-σ: {sigma_delta_x:.4f} m')
ax1.set_xlabel('Delta X (m)')
ax1.set_ylabel('Count')
ax1.set_title('Delta X Distribution (Combined Product Loss)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Histogram of |delta_X|
abs_delta_x = np.abs(valid_delta_x)
ax2.hist(abs_delta_x, bins=25, alpha=0.7, edgecolor='black')
percentile_68 = np.percentile(abs_delta_x, 68)
ax2.axvline(percentile_68, color='red', linestyle='--', label=f'68th percentile: {percentile_68:.4f} m')
ax2.set_xlabel('|Delta X| (m)')
ax2.set_ylabel('Count')
ax2.set_title('|Delta X| Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"68% of |delta_X| values lie between 0 and {percentile_68:.4f} m")
print(f"Expected 3D resolution: {percentile_68 * np.sqrt(3):.4f} m")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# Calculate Euclidean distance to detector center for each event
distances = []
x_components = []
for event in all_results['events']:
    pos = np.array(event['true_position'])
    distance = np.linalg.norm(pos)  # Distance from [0,0,0]
    distances.append(distance)
    x_components.append(np.array(event['true_direction'][0]))

distances = np.array(distances)
x_components = np.array(x_components)
valid_distances = distances[~np.isnan(delta_x_array)]
valid_x_components = x_components[~np.isnan(delta_x_array)]

# --- ΔX vs Distance ---
slope_dist, intercept_dist, r_dist, _, _ = linregress(valid_distances, valid_delta_x)
fit_dist = slope_dist * valid_distances + intercept_dist
dist_to_line_dist = np.abs(slope_dist * valid_distances - valid_delta_x + intercept_dist) / np.sqrt(slope_dist**2 + 1)
sigma68_dist = np.percentile(dist_to_line_dist, 68)

# --- ΔX vs X-direction component ---
slope_xdir, intercept_xdir, r_xdir, _, _ = linregress(valid_x_components, valid_delta_x)
fit_xdir = slope_xdir * valid_x_components + intercept_xdir
dist_to_line_xdir = np.abs(slope_xdir * valid_x_components - valid_delta_x + intercept_xdir) / np.sqrt(slope_xdir**2 + 1)
sigma68_xdir = np.percentile(dist_to_line_xdir, 68)

# --- Plot both ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# ΔX vs Distance
ax1.scatter(valid_distances, valid_delta_x, alpha=0.7, s=50, label='Data')
ax1.plot(valid_distances, fit_dist, color='red', label=f'Fit: y={slope_dist:.3f}x+{intercept_dist:.3f}')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
ax1.set_xlabel('Distance to Detector Center (m)')
ax1.set_ylabel('Delta X (m)')
ax1.set_title('ΔX vs Distance to Center')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.text(0.05, 0.95,
         f'Slope = {slope_dist:.3f}\nIntercept = {intercept_dist:.3f}\nR = {r_dist:.3f}\nσ₆₈ = {sigma68_dist:.4f}',
         transform=ax1.transAxes,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# ΔX vs X-direction component
ax2.scatter(valid_x_components, valid_delta_x, alpha=0.7, s=50, label='Data')
ax2.plot(valid_x_components, fit_xdir, color='red', label=f'Fit: y={slope_xdir:.3f}x+{intercept_xdir:.3f}')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
ax2.set_xlabel('X-Direction Component')
ax2.set_ylabel('Delta X (m)')
ax2.set_title('ΔX vs X-Direction Component')
ax2.grid(True, alpha=0.3)
ax2.legend()
ax2.text(0.05, 0.95,
         f'Slope = {slope_xdir:.3f}\nIntercept = {intercept_xdir:.3f}\nR = {r_xdir:.3f}\nσ₆₈ = {sigma68_xdir:.4f}',
         transform=ax2.transAxes,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

# --- Print summary ---
print("=== Linear Fit Results ===")
print(f"ΔX vs Distance:")
print(f"  Slope: {slope_dist:.4f}")
print(f"  Intercept: {intercept_dist:.4f}")
print(f"  R-value: {r_dist:.4f}")
print(f"  σ₆₈: {sigma68_dist:.4f} m\n")

print(f"ΔX vs X-Direction Component:")
print(f"  Slope: {slope_xdir:.4f}")
print(f"  Intercept: {intercept_xdir:.4f}")
print(f"  R-value: {r_xdir:.4f}")
print(f"  σ₆₈: {sigma68_xdir:.4f} m")


In [ ]:
def perform_t0_scan_combined_loss(true_track, true_data, key, t0_scan_range=5.0, verbose=False):
    """
    Perform 1D scan over t0 using combined product loss function

    Args:
        true_track: ParticleParams object
        true_data: target data-like event (hit_counts, hit_times)
        key: JAX random key
        t0_scan_range: range to scan around true t0 (+/-range in ns)
        verbose: if True, print detailed timing information

    Returns:
        dict with t0_values, losses, gradients, and timing info
    """
    t_start_total = time.time()

    # Setup phase
    t_start_setup = time.time()
    true_energy, true_position, true_direction = true_track.energy, true_track.position, true_track.direction
    true_t0 = 0.0  # Default t0

    # Extract hit times from true_data
    hit_counts, hit_times = true_data

    # Filter for detectors that were hit
    hit_mask = hit_counts > -1
    hit_detector_positions = detector_points[hit_mask]
    observed_times = hit_times[hit_mask]
    observed_charge = hit_counts[hit_mask]

    # Generate t0 scan points
    t0_values = jnp.linspace(true_t0 - t0_scan_range, true_t0 + t0_scan_range, N_SCAN_POINTS)

    # Convert true direction to spherical coordinates
    from lucid.optimization.utils.functions import cartesian_to_spherical
    true_theta, true_phi = cartesian_to_spherical(true_direction)

    t_setup = time.time() - t_start_setup

    # Define loss and gradient function
    def loss_and_grad_fn(params):
        def loss_fn(p):
            # Need a random key for simulator in combined_product_loss
            loss_key = jax.random.PRNGKey(42)  # Fixed key for consistency
            return combined_product_loss(p, hit_detector_positions, observed_times, observed_charge,
                                        true_data, loss_key)
        return value_and_grad(loss_fn)(params)

    # Warmup (JIT compilation) - run once before timing
    t_start_warmup = time.time()
    warmup_params = jnp.array([true_position[0], true_position[1], true_position[2], 
                               true_t0, true_theta, true_phi, true_energy])
    _ = loss_and_grad_fn(warmup_params)
    jax.block_until_ready(_)  # Wait for computation to complete
    t_warmup = time.time() - t_start_warmup

    # Scan loop
    losses = []
    gradients = []
    per_point_times = []

    t_start_scan = time.time()
    for i, t0_val in enumerate(t0_values):
        t_point_start = time.time()

        # Create parameter vector with modified t0
        # params: [x, y, z, t0, theta, phi, energy]
        params = jnp.array([
            true_position[0],  # x
            true_position[1],  # y
            true_position[2],  # z
            t0_val,            # modified t0
            true_theta,        # theta
            true_phi,          # phi
            true_energy        # energy
        ])

        # Calculate loss and gradient
        t_loss_grad_start = time.time()
        loss, grad_val = loss_and_grad_fn(params)
        jax.block_until_ready((loss, grad_val))  # Ensure computation completes
        t_loss_grad = time.time() - t_loss_grad_start

        # Extract t0 gradient
        t0_gradient = grad_val[3]  # gradient w.r.t. params[3] (t0)

        losses.append(loss)
        gradients.append(t0_gradient)

        t_point = time.time() - t_point_start
        per_point_times.append(t_point)

        if verbose and i == 0:
            print(f"    First point timing: {t_point:.4f}s (loss+grad: {t_loss_grad:.4f}s)")

    t_scan = time.time() - t_start_scan
    t_total = time.time() - t_start_total

    # Timing statistics
    per_point_times = np.array(per_point_times)
    timing_info = {
        'total_time': t_total,
        'setup_time': t_setup,
        'warmup_time': t_warmup,
        'scan_time': t_scan,
        'avg_per_point': np.mean(per_point_times),
        'std_per_point': np.std(per_point_times),
        'min_per_point': np.min(per_point_times),
        'max_per_point': np.max(per_point_times),
        'first_point_time': per_point_times[0],
        'subsequent_avg': np.mean(per_point_times[1:]) if len(per_point_times) > 1 else 0,
    }

    if verbose:
        print(f"\n  Timing breakdown:")
        print(f"    Setup:              {timing_info['setup_time']:.4f}s")
        print(f"    Warmup (JIT):       {timing_info['warmup_time']:.4f}s")
        print(f"    Scan loop:          {timing_info['scan_time']:.4f}s")
        print(f"    Total:              {timing_info['total_time']:.4f}s")
        print(f"    Per point (avg):    {timing_info['avg_per_point']:.4f}s +/- {timing_info['std_per_point']:.4f}s")
        print(f"    First point:        {timing_info['first_point_time']:.4f}s")
        print(f"    Subsequent (avg):   {timing_info['subsequent_avg']:.4f}s")

    return {
        't0_values': jnp.array(t0_values),
        'losses': jnp.array(losses),
        'gradients': jnp.array(gradients),
        'n_hit_detectors': jnp.sum(hit_mask),
        'timing': timing_info
    }

print("t0 scan function defined (with timing)")

In [ ]:
all_results = {
    'c_medium': C_MEDIUM,
    'temperature': TEMPERATURE,
    'n_events': N_EVENTS,
    'n_scan_points': N_SCAN_POINTS,
    'events': [],
    'timing_stats': []  # Store timing for each event
}

print(f"Analyzing {N_EVENTS} data-like events with combined product loss t0 scans...")

# Generate random keys for each event
main_key = jax.random.PRNGKey(42)
event_keys = jax.random.split(main_key, N_EVENTS)

for event_idx in tqdm(range(N_EVENTS), desc="Processing events"):
    # Select entry from ROOT file (cycling through available entries)
    entry_idx = event_idx % n_entries
    
    # Load photon data from ROOT file
    photon_data = read_photon_data_from_photonsim(data_file, entry_idx)

    # Process photon data
    photon_origins = photon_data['photon_origins']
    photon_directions = photon_data['photon_directions']
    photon_times = photon_data['photon_times']
    N = len(photon_origins)

    # the number 1_000_000 is hard coded also in _simulation_core
    padding_size = max(0, 1_000_000-N)

    # Pad the origins array (2D array with shape [N,3])
    photon_data['photon_origins'] = jnp.pad(photon_origins, ((0, padding_size), (0, 0)), 
                                        mode='constant', constant_values=0)

    # Pad the directions array with a default unit vector [0,0,1]
    default_direction = jnp.array([0.0, 0.0, 1.0])
    padding_directions = jnp.tile(default_direction, (padding_size, 1))
    if padding_size > 0:
        photon_data['photon_directions'] = jnp.concatenate([photon_directions, padding_directions], axis=0)
    else:
        photon_data['photon_directions'] = photon_directions

    # Pad the times array (1D array with shape [N])
    photon_data['photon_times'] = jnp.pad(photon_times, (0, padding_size),
                                          mode='constant', constant_values=0)

    photon_data['N'] = N
    
    # Generate random track parameters
    key = event_keys[event_idx]
    
    # Random position within detector bounds (60% of full volume)
    fraction = 0.6
    r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=detector.r * fraction)
    key, _ = jax.random.split(key)
    theta = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    z_vert = jax.random.uniform(key, shape=(), minval=-detector.H/2 * fraction, 
                               maxval=detector.H/2 * fraction)
    true_position = jnp.array([r_vert * jnp.cos(theta), r_vert * jnp.sin(theta), z_vert])
    
    # Random direction
    key, _ = jax.random.split(key)
    phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
    sin_theta = jnp.sqrt(1 - cos_theta**2)
    true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

    original_direction = jnp.array([0.0, 0.0, 1.0])
    true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)
    
    # Rotation axis = cross product of original and target directions
    rotation_axis = jnp.cross(original_direction, true_direction_norm)
    axis_norm = jnp.linalg.norm(rotation_axis)
    
    # Handle case where directions are parallel (axis_norm ~ 0)
    rotation_axis = jnp.where(
        axis_norm < 1e-6,
        jnp.array([1.0, 0.0, 0.0]),  # Arbitrary axis when parallel
        rotation_axis / (axis_norm + 1e-8)
    )
    
    # Rotation angle = arccos of dot product
    rotation_angle = jnp.arccos(jnp.clip(
        jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
    ))
    
    # Set rotation parameters
    photon_data['rotation_axis'] = rotation_axis
    photon_data['rotation_angle'] = rotation_angle
    photon_data['apply_rotation'] = jnp.array(True)
    
    # Set translation parameters to move from origin to true_position
    photon_data['apply_translation'] = jnp.array(True)
    photon_data['translation_vector'] = true_position
    
    # Use energy from ROOT file
    true_energy = photon_data['energy']  # Fixed energy
    
    # Create particle parameters
    true_track = ParticleParams.from_cartesian(energy=true_energy, position=true_position, direction=true_direction, t0=0.0)
    
    # Generate data-like event
    key, _ = jax.random.split(key)
    before_data_calc = time.time()
    true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))
    
    # Perform t0 scan with combined product loss
    # Enable verbose for first event only
    key, _ = jax.random.split(key)
    scan_results = perform_t0_scan_combined_loss(true_track, true_data, key, verbose=(event_idx < 3))
    
    # Store results for this event
    event_result = {
        'event_idx': event_idx,
        'entry_idx': entry_idx,
        'true_energy': float(true_energy),
        'true_position': [float(x) for x in true_position],
        'true_direction': [float(x) for x in true_direction],
        'true_t0': 0.0,  # Default t0
        't0_values': [float(x) for x in scan_results['t0_values']],
        'losses': [float(x) for x in scan_results['losses']],
        'gradients': [float(x) for x in scan_results['gradients']],
        'n_hit_detectors': int(scan_results['n_hit_detectors'])
    }
    
    all_results['events'].append(event_result)
    all_results['timing_stats'].append(scan_results['timing'])
    
    if event_idx == 0:
        print(f"\nEvent {event_idx} (entry {entry_idx}):")
        print(f"  Energy: {true_energy:.2f} MeV")
        print(f"  Position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}] m")
        print(f"  Direction: [{true_direction[0]:.3f}, {true_direction[1]:.3f}, {true_direction[2]:.3f}]")
        print(f"  t0 scan range: [{scan_results['t0_values'][0]:.2f}, {scan_results['t0_values'][-1]:.2f}] ns")
        print(f"  Hit detectors: {scan_results['n_hit_detectors']}")

print(f"\nCompleted analysis of {N_EVENTS} events with {N_SCAN_POINTS} t0 points each.")

In [ ]:
delta_t0_values = []

for event in all_results['events']:
    t0_values = np.array(event['t0_values'])
    gradients = np.array(event['gradients'])
    true_t0 = event['true_t0']

    zero_cross_found = False

    for i in range(len(gradients) - 1):
        g1, g2 = gradients[i], gradients[i + 1]

        # Check for a sign change (crossing zero)
        if g1 == 0:
            zero_grad_t0 = t0_values[i]
            zero_cross_found = True
            break
        elif g1 * g2 < 0:  # gradient crosses zero between these points
            t1, t2 = t0_values[i], t0_values[i + 1]

            # Linear interpolation: t0_zero = t1 - g1 * (t2 - t1) / (g2 - g1)
            zero_grad_t0 = t1 - g1 * (t2 - t1) / (g2 - g1)
            zero_cross_found = True
            break

    if zero_cross_found:
        delta_t0 = zero_grad_t0 - true_t0
        delta_t0_values.append(delta_t0)
    else:
        delta_t0_values.append(np.nan)

# Calculate statistics
delta_t0_array = np.array(delta_t0_values)
valid_delta_t0 = delta_t0_array[~np.isnan(delta_t0_array)]
mean_delta_t0 = np.mean(valid_delta_t0)
sigma_delta_t0 = np.std(valid_delta_t0)

print(f"Number of events with zero-crossing gradients: {len(valid_delta_t0)}")
print(f"Number of events without zero-crossing gradients: {len(delta_t0_array) - len(valid_delta_t0)}")
print(f"Mean delta_t0: {mean_delta_t0:.4f} ns")
print(f"Sigma delta_t0: {sigma_delta_t0:.4f} ns")

In [ ]:
# Plot results for first few events
n_plot_events = min(10, N_EVENTS)

fig, axes = plt.subplots(2, n_plot_events, figsize=(4*n_plot_events, 8))
if n_plot_events == 1:
    axes = axes.reshape(2, 1)

for i in range(n_plot_events):
    event = all_results['events'][i]
    t0_values = np.array(event['t0_values'])
    losses = np.array(event['losses'])
    gradients = np.array(event['gradients'])
    true_t0 = event['true_t0']
    n_hit = event['n_hit_detectors']
    
    # Plot loss
    axes[0, i].plot(t0_values, losses, 'b-', linewidth=2)
    axes[0, i].axvline(true_t0, color='red', linestyle='--', alpha=0.7, label='True t0')
    
    # Find and mark minimum loss position
    min_loss_idx = np.argmin(losses)
    min_loss_t0 = t0_values[min_loss_idx]
    axes[0, i].axvline(min_loss_t0, color='orange', linestyle=':', alpha=0.7, label='Min Loss')
    
    axes[0, i].set_title(f'Event {i} - Combined Product Loss\n({n_hit} hit detectors)')
    axes[0, i].set_xlabel('t0 (ns)')
    axes[0, i].set_ylabel('Loss')
    axes[0, i].grid(True, alpha=0.3)
    axes[0, i].legend()
    
    # Plot gradient
    axes[1, i].plot(t0_values, gradients, 'g-', linewidth=2)
    axes[1, i].axvline(true_t0, color='red', linestyle='--', alpha=0.7, label='True t0')
    axes[1, i].axhline(0, color='gray', linestyle=':', alpha=0.5)
    
    # Mark zero gradient crossing if it exists
    if not np.isnan(delta_t0_values[i]):
        zero_grad_t0 = true_t0 + delta_t0_values[i]
        axes[1, i].axvline(zero_grad_t0, color='purple', linestyle='-.', alpha=0.7, label='Zero Grad')
    
    axes[1, i].set_title(f'Event {i} - Gradient')
    axes[1, i].set_xlabel('t0 (ns)')
    axes[1, i].set_ylabel('dLoss/dt0')
    axes[1, i].grid(True, alpha=0.3)
    axes[1, i].legend()

plt.tight_layout()
plt.show()

print(f"Plotted results for first {n_plot_events} events.")

In [ ]:
# Plot histogram of delta_t0 values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Histogram of delta_t0
ax1.hist(valid_delta_t0, bins=25, alpha=0.7, edgecolor='black')
ax1.axvline(mean_delta_t0, color='red', linestyle='--', label=f'Mean: {mean_delta_t0:.4f} ns')
ax1.axvline(mean_delta_t0 + sigma_delta_t0, color='orange', linestyle=':', label=f'+sigma: {sigma_delta_t0:.4f} ns')
ax1.axvline(mean_delta_t0 - sigma_delta_t0, color='orange', linestyle=':', label=f'-sigma: {sigma_delta_t0:.4f} ns')
ax1.set_xlabel('Delta t0 (ns)')
ax1.set_ylabel('Count')
ax1.set_title('Delta t0 Distribution (Combined Product Loss)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Histogram of |delta_t0|
abs_delta_t0 = np.abs(valid_delta_t0)
ax2.hist(abs_delta_t0, bins=25, alpha=0.7, edgecolor='black')
percentile_68 = np.percentile(abs_delta_t0, 68)
ax2.axvline(percentile_68, color='red', linestyle='--', label=f'68th percentile: {percentile_68:.4f} ns')
ax2.set_xlabel('|Delta t0| (ns)')
ax2.set_ylabel('Count')
ax2.set_title('|Delta t0| Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"68% of |delta_t0| values lie between 0 and {percentile_68:.4f} ns")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# Calculate Euclidean distance to detector center for each event
distances = []
for event in all_results['events']:
    pos = np.array(event['true_position'])
    distance = np.linalg.norm(pos)  # Distance from [0,0,0]
    distances.append(distance)

distances = np.array(distances)
valid_distances = distances[~np.isnan(delta_t0_array)]

# --- Delta t0 vs Distance ---
slope_dist, intercept_dist, r_dist, _, _ = linregress(valid_distances, valid_delta_t0)
fit_dist = slope_dist * valid_distances + intercept_dist
dist_to_line_dist = np.abs(slope_dist * valid_distances - valid_delta_t0 + intercept_dist) / np.sqrt(slope_dist**2 + 1)
sigma68_dist = np.percentile(dist_to_line_dist, 68)

# --- Plot ---
fig, ax1 = plt.subplots(1, 1, figsize=(8, 6))

# Delta t0 vs Distance
ax1.scatter(valid_distances, valid_delta_t0, alpha=0.7, s=50, label='Data')
ax1.plot(valid_distances, fit_dist, color='red', label=f'Fit: y={slope_dist:.3f}x+{intercept_dist:.3f}')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
ax1.set_xlabel('Distance to Detector Center (m)')
ax1.set_ylabel('Delta t0 (ns)')
ax1.set_title('Delta t0 vs Distance to Center')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.text(0.05, 0.95,
         f'Slope = {slope_dist:.3f}\nIntercept = {intercept_dist:.3f}\nR = {r_dist:.3f}\nsigma_68 = {sigma68_dist:.4f}',
         transform=ax1.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

# --- Print summary ---
print("=== Linear Fit Results ===")
print(f"Delta t0 vs Distance:")
print(f"  Slope: {slope_dist:.4f}")
print(f"  Intercept: {intercept_dist:.4f}")
print(f"  R-value: {r_dist:.4f}")
print(f"  sigma_68: {sigma68_dist:.4f} ns")